In [1]:
from sys import prefix
from typing import Any

"""
数值梯度检查
"""

from collections.abc import Callable
import dnnlpy.models.mlp as mlp
import numpy as np
import numpy.linalg as npl
import torch
import torch.autograd as AF
from torch import Tensor

type Func = Callable[[np.ndarray], np.ndarray]

rng = np.random.default_rng(42)
print("PyTorch Version: ", torch.__version__)


PyTorch Version:  2.13.0+cpu


In [6]:
"""
数值梯度检查的小例子
"""


def numerical_gradient(func: Func, x: np.ndarray, esp: float = 1e-5) -> np.ndarray:
    grad = np.zeros_like(x)

    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        old_val = x[idx]

        x[idx] = old_val + esp
        fx_plus = func(x)

        x[idx] = old_val - esp
        fx_minus = func(x)

        x[idx] = old_val
        grad[idx] = (fx_plus - fx_minus) / (2 * esp)
        it.iternext()
    return grad


def func(x: np.ndarray) -> float:
    return np.sum(np.square(x))


def relative_error(analytic: np.ndarray, numeric: np.ndarray, eps: float = 1e-12) -> float:
    """
    计算误差
    :param analytic:
    :param numeric:
    :param eps:
    :return:
    """
    a = npl.norm(analytic - numeric)
    b = npl.norm(analytic) + npl.norm(numeric) + eps
    return a / b


x = np.array([1.0, 2.0, -2.0, 3.0])
grad_numerical = numerical_gradient(func, x)
grad_analytic = 2 * x
print('numerical gradient:', grad_numerical)
print('Analytic gradient:', grad_analytic)

error = relative_error(grad_analytic, grad_numerical)
print('Relative error:', error)

numerical gradient: [ 2.  4. -4.  6.]
Analytic gradient: [ 2.  4. -4.  6.]
Relative error: 9.891236144011368e-12


In [13]:
"""
检查Linear的backward
"""

x = rng.standard_normal((4, 3))
linear = mlp.Linear(in_features=3, out_features=2)

#手动计算backward  得到解析梯度
out = linear(x)
loss = np.sum(np.square(out))
dout = 2 * out
grad_x_analytic = linear.backward(dout)

grad_W_analytic = linear.weight.grad.copy()
grad_b_analytic = linear.bias.grad.copy()


# 使用数值检查
def loss_fn_weight(weight: np.ndarray) -> float:
    old_weight = linear.weight
    linear.weight = weight
    out = linear(x)
    loss = np.sum(np.square(out))

    linear.weight = old_weight
    return loss


grad_W_numerical = numerical_gradient(loss_fn_weight, linear.weight.copy())
error = relative_error(grad_W_analytic, grad_W_numerical)
print('Relative error for w:', error)


# 检查b

def loss_fn_bias(bias: np.ndarray) -> float:
    old_bias = linear.bias
    linear.bias = bias
    out = linear(x)
    loss = np.sum(np.square(out))
    linear.bias = old_bias
    return loss


grad_bias_numeric = numerical_gradient(loss_fn_bias, linear.bias.copy())
error = relative_error(grad_b_analytic, grad_bias_numeric)
print('Relative error for b:', error)


# 检查x
def loss_fn_x(x: np.ndarray) -> float:
    out = linear(x)
    loss = np.sum(np.square(out))
    return loss


grad_x_numeric = numerical_gradient(loss_fn_x, x.copy())
error = relative_error(grad_x_analytic, grad_x_numeric)
print('Relative error for x:', error)



Relative error for w: 0.0006785631924451269
Relative error for b: 3.196461491846563e-08
Relative error for x: 7.376923080121744e-12


In [14]:
"""
检查ReLU 的backward
"""
x = rng.standard_normal((4, 5))
relu = mlp.ReLU()

out = relu(x)
loss = np.sum(out ** 2)
dout = 2 * out
grad_x_analytic = relu.backward(dout)


def loss_fn_relu(x: np.ndarray) -> float:
    out = relu(x)
    loss = np.sum(np.square(out))
    return loss


grad_x_numeric = numerical_gradient(loss_fn_relu, x.copy())
error = relative_error(grad_x_analytic, grad_x_numeric)
print("Relative error for relu:", error)

Relative error for relu: 5.620349661937006e-12


In [16]:
"""
检查softmax cross entropy  的把backward
"""
logits = rng.standard_normal((4, 3))
y = np.array([0, 2, 1, 2])

loss_fn = mlp.CrossEntropyLoss()
loss = loss_fn(logits, y)
grad_logits_analytic = loss_fn.backward()


def loss_fn_logits(logits: np.ndarray) -> float:
    loss = loss_fn(logits, y)
    return loss


grad_logits_numeric = numerical_gradient(loss_fn_logits, logits.copy())
error = relative_error(grad_logits_analytic, grad_logits_numeric)
print("Relative error for logits:", error)

Relative error for logits: 2.6728751619954295e-11


In [21]:
"""
检查完整的MLP的参数梯度
"""
x = rng.standard_normal((5, 4))
y = np.array([0, 1, 2, 1, 0])

model = mlp.MLP(input_dim=4, hidden_dim=6, num_classes=3)
loss_fn = mlp.CrossEntropyLoss()

# backward 解析梯度
logits = model(x)
loss = loss_fn(logits, y)
dlogits = loss_fn.backward()
dx = model.backward(dlogits)

grad_fc1_W_analytic = model.fc1.weight.grad.copy()


# 使用数值估计fc1.weight 的梯度
def loss_fn_weights(weight: np.ndarray) -> float:
    old_weight = model.fc1.weight.copy()
    model.fc1.weight = weight
    logits = model(x)
    loss = loss_fn(logits, y)
    model.fc1.weight = old_weight
    return loss


grad_fc1_W_numeric = numerical_gradient(loss_fn_weights, model.fc1.weight.copy())
error = relative_error(grad_fc1_W_analytic, grad_fc1_W_numeric)
print("Relative error for weights:", error)



Relative error for weights: 0.0005707559301731147


In [29]:
"""
PyTorch  中的 Gradient  Check
"""


class ReLU(AF.Function):

    @staticmethod
    def forward(ctx: AF.Function, x: Tensor) -> Tensor:
        ctx.save_for_backward(x)
        return x.relu()

    @staticmethod
    def backward(ctx: AF.Function, grad_output: Tensor) -> Tensor:
        x = ctx.saved_tensors[0]
        return grad_output * (x > 0).double()


def relu(x: Tensor) -> Tensor:
    return ReLU.apply(x)


x = torch.randn(10, dtype=torch.double, requires_grad=True)
flag = AF.gradcheck(relu, (x,))
print("Gradient check passed:", flag)

Gradient check passed: True
